In [6]:
# --- Cell 1: load data ---
import re
import pandas as pd
import os
from collections import defaultdict
from difflib import get_close_matches

notebook_path = os.getcwd()
in_dir_notifications_raw = os.path.abspath(os.path.join(notebook_path, "..", "..", "Data", "notification_historical", "notification.jsonl"))
in_dir_master_direction = os.path.abspath(os.path.join(notebook_path, "..", "..", "Data", "master_directory", "master_directory.jsonl"))

circulars = pd.read_json(in_dir_notifications_raw, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines=True)
print(f"circulars: {len(circulars)}, master_directions: {len(master_directions)}")

circulars: 766, master_directions: 44


In [7]:
# --- Cell 2: citation regex + normalizer ---
DOC_TYPE = r"(?:Directions?|Guidelines?|Regulations?|Rules?|Circulars?|Framework|Scheme)"
p1 = re.compile(
    r"Reserve Bank of India\s*[-–—]?\s*\(([^)]+)\)"
    r"(?:\s*\([^)]*\))*"
    r"\s*(?:[A-Za-z]+\s+){0,3}" + DOC_TYPE,
    re.IGNORECASE
)
p2 = re.compile(r"Reserve Bank of India\s*[-–—]\s*([A-Za-z ,]+?),?\s*Directions", re.IGNORECASE)
norm = lambda s: re.sub(r"\s+", " ", s.replace("–", "-").replace("—", "-")).strip(" ,-").lower()

def extract_names(text):
    if not isinstance(text, str):
        return []
    return [norm(x) for x in (p1.findall(text) + p2.findall(text))]

In [19]:
# --- Cell 3: title + "lead paragraph" extraction ---
# cuts text at the start of numbered paragraph "2." (RBI's own background/operative boundary)
# falls back to first 600 chars if no numbered paragraph is found
lead_break = re.compile(r"\n\s*2\.\s")

def get_lead(text):
    if not isinstance(text, str):
        return ""
    m = lead_break.search(text)
    return text[:m.start()] if m else text[:600]
# --- Cell 3 (revised) ---
circulars["found_text"]      = circulars["text"].apply(extract_names)          # for matching
circulars["found_text_lead"] = circulars["text"].apply(lambda t: extract_names(get_lead(t)))  # for bucketing only

In [20]:
# --- Cell 4: build master_lookup, flagging collisions instead of silently overwriting ---
master_directions["extracted_names"] = master_directions["title"].apply(extract_names)

name_to_ids = defaultdict(set)
for _, r in master_directions.iterrows():
    for name in r["extracted_names"]:
        name_to_ids[name].add(r["id"])

collisions = {k: v for k, v in name_to_ids.items() if len(v) > 1}
print(f"{len(collisions)} names collide across master_directions (excluded from lookup):")
print(collisions)

unlinkable = master_directions[master_directions["extracted_names"].str.len() == 0]
print(f"{len(unlinkable)}/{len(master_directions)} master directions have no extractable name")

# only keep unambiguous names
master_lookup = {name: next(iter(ids)) for name, ids in name_to_ids.items() if len(ids) == 1}

1 names collide across master_directions (excluded from lookup):
{'non-banking financial companies - miscellaneous': {13586, 12931}}
0/44 master directions have no extractable name


In [24]:

# --- Cell 5 (revised) ---
def match_row(row):
    for n in row["found_title"]:
        if n in master_lookup:
            return master_lookup[n], "title_match"
    for n in row["found_text"]:          # full text, not lead
        if n in master_lookup:
            return master_lookup[n], "text_match"
    return None, "no_match"

circulars[["matched_id", "match_method"]] = circulars.apply(lambda r: pd.Series(match_row(r)), axis=1)
print(circulars["match_method"].value_counts())

match_method
no_match       665
title_match     76
text_match      25
Name: count, dtype: int64


In [26]:
recovered_by_full_text = circulars[
    (circulars["match_method"] != "no_match") &
    (circulars["found_title"].str.len() == 0) &
    (~circulars["found_text_lead"].apply(lambda lst: any(n in master_lookup for n in lst)))
]
print(len(recovered_by_full_text))
recovered_by_full_text[["id", "title", "matched_id"]]

1


,id,title,matched_id
244,13169,Compliance with Know Your Customer (KYC) norms,12943.0


In [25]:
# --- Cell 6, revised ---
from difflib import SequenceMatcher

def char_diff(a, b):
    diff = 0
    for tag, i1, i2, j1, j2 in SequenceMatcher(None, a, b).get_opcodes():
        if tag != "equal":
            diff += max(i2 - i1, j2 - j1)
    return diff
fuzzy_candidates = []
for idx, row in circulars[circulars["match_method"] == "no_match"].iterrows():
    for n in (row["found_title"] + row["found_text_lead"]):
        for key in master_keys:
            if char_diff(n, key) <= 5:   # small typo/formatting gap only, not a different phrase
                fuzzy_candidates.append((idx, row["id"], n, key))
                break

fuzzy_df = pd.DataFrame(fuzzy_candidates, columns=["row_idx", "circular_id", "extracted_name", "closest_master_name"])
print(f"{len(fuzzy_df)} candidates")
fuzzy_df

3 candidates


,row_idx,circular_id,extracted_name,closest_master_name
0,288,13213,non-operative financial holding company,non-operative financial holding companies
1,288,13213,non-operative financial holding company,non-operative financial holding companies
2,454,13379,non-banking financial companies- undertaking of financial services,non-banking financial companies - undertaking of financial services


In [27]:
# --- Cell 6b: apply fuzzy matches you've reviewed and trust ---
for _, r in fuzzy_df.iterrows():
    circulars.loc[r["row_idx"], "matched_id"]    = master_lookup[r["closest_master_name"]]
    circulars.loc[r["row_idx"], "match_method"]  = "fuzzy_match"

print(circulars["match_method"].value_counts())

match_method
no_match       663
title_match     76
text_match      25
fuzzy_match      2
Name: count, dtype: int64


In [28]:
# --- Cell 7: NBFC relevance + final bucketing ---
nbfc_pattern = r"non[\s-]?banking financial compan|nbfc"
circulars["is_nbfc_relevant"] = (
    circulars["text"].str.contains(nbfc_pattern, case=False, na=False) |
    circulars["title"].str.contains(nbfc_pattern, case=False, na=False)
)

def bucket(row):
    if row["match_method"] != "no_match":
        return "linked"
    if not row["is_nbfc_relevant"]:
        return "not_nbfc"
    has_citation = len(row["found_title"]) > 0 or len(row["found_text_lead"]) > 0
    return "nbfc_citation_unmatched" if has_citation else "nbfc_standalone"

circulars["bucket"] = circulars.apply(bucket, axis=1)
print(circulars["bucket"].value_counts())

bucket
not_nbfc                   529
nbfc_citation_unmatched    133
linked                     103
nbfc_standalone              1
Name: count, dtype: int64


In [29]:
# --- Cell 8: subject_code as an independent cross-check (not a primary matcher) ---
ref_pattern = re.compile(r"([A-Z]+(?:\.[A-Z]+)+\.\d+)/([\d-]+)/(\d{4}-\d{2})")
circulars["subject_code"] = circulars["text"].str.extract(ref_pattern)[1]
master_directions["subject_code"] = master_directions["text"].str.extract(ref_pattern)[1]

# auto-drop any code shared by more than one master direction (generic/bucket codes)
code_counts = master_directions.dropna(subset=["subject_code"]).groupby("subject_code")["id"].nunique()
generic_codes = code_counts[code_counts > 1].index.tolist()
clean_master = master_directions[~master_directions["subject_code"].isin(generic_codes)]

code_matches = circulars.merge(
    clean_master.dropna(subset=["subject_code"])[["id", "subject_code"]],
    on="subject_code", how="left", suffixes=("", "_master")
)
both = code_matches[(code_matches["match_method"] != "no_match") & code_matches["id_master"].notna()]
disagreements = both[both["matched_id"] != both["id_master"]]
print(f"{len(disagreements)} / {len(both)} disagree between regex-match and code-match")

0 / 7 disagree between regex-match and code-match


In [30]:
# --- Cell 9: final tally ---
print(circulars["bucket"].value_counts())
nbfc_total = circulars["bucket"].isin(["linked", "nbfc_citation_unmatched", "nbfc_standalone"]).sum()
print(f"NBFC-relevant: {nbfc_total} / {len(circulars)}")
print(f"Not NBFC: {(circulars['bucket']=='not_nbfc').sum()} / {len(circulars)}")

bucket
not_nbfc                   529
nbfc_citation_unmatched    133
linked                     103
nbfc_standalone              1
Name: count, dtype: int64
NBFC-relevant: 237 / 766
Not NBFC: 529 / 766


In [31]:
unresolved = circulars[circulars["bucket"] == "nbfc_citation_unmatched"]
cited_names = unresolved["found_title"].apply(lambda l: l[0] if l else None).dropna()
cited_names.value_counts().head(20)

found_title
regional rural banks - resolution of stressed assets                                                  2
local area banks - resolution of stressed assets                                                      2
small finance banks - resolution of stressed assets                                                   2
commercial banks - resolution of stressed assets                                                      2
commercial banks - undertaking of financial services                                                  2
commercial banks - credit facilities                                                                  2
small finance banks - credit facilities                                                               2
credit information companies - managing risks in outsourcing                                          1
credit information companies                                                                          1
asset reconstruction companies - know your customer 